# Part 12: 엔터프라이즈 GraphRAG**소요 시간**: 약 1.5시간**난이도**: ⭐⭐⭐⭐⭐ (고급)**목표**: ROI 분석, PoC→프로덕션 전환, 보안/컴플라이언스, CI/CD 운영 자동화를 설계할 수 있다.---## 학습 순서1. 환경 설정2. ROI 분석 + 비즈니스 케이스3. PoC → 프로덕션 전환4. 보안 & 컴플라이언스5. CI/CD + 운영 자동화6. 확장 전략7. 연습 문제

---## 1. 환경 설정### 패키지 설치 및 Neo4j 연결

In [ ]:
import os, json, timefrom dotenv import load_dotenvfrom neo4j import GraphDatabaseload_dotenv()load_dotenv(dotenv_path="../.env")NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))try:    driver.verify_connectivity()    print("[OK] Neo4j 연결 성공!")except Exception as e:    print(f"[ERROR] 연결 실패: {e}")def run_query(query, parameters=None, print_result=True):    with driver.session() as session:        result = session.run(query, parameters or {})        records = [record.data() for record in result]        if print_result:            for i, rec in enumerate(records, 1):                print(f"  [{i}] {rec}")            if not records:                print("  (결과 없음)")        return recordsprint("[OK] 환경 설정 완료")

---## 2. ROI 분석 + 비즈니스 케이스### GraphRAG 도입 비용 구조| 항목 | 초기 비용 | 월 운영비 | 비고 ||------|---------|----------|------|| Neo4j (Community) | $0 | $0 | 오픈소스 || Neo4j (Enterprise) | - | $2K~$50K | 클러스터, 보안 || LLM API | - | $500~$5K | GPT-4 기준 || 인프라 (클라우드) | - | $200~$2K | GPU 불필요 || 개발 인력 | $20K~$50K | - | 1-2명, 3개월 |### ROI 계산 공식```ROI = (연간 절감액 - 연간 비용) / 연간 비용 × 100%예시:  절감: 고객 응답 자동화로 연 $120K 절감  비용: $60K (개발 $30K + 운영 $30K)  ROI = (120K - 60K) / 60K × 100% = 100%```

In [ ]:
# ROI 계산기def calculate_roi(annual_savings, initial_cost, monthly_cost):    annual_cost = initial_cost + (monthly_cost * 12)    roi = (annual_savings - annual_cost) / annual_cost * 100    payback_months = initial_cost / (annual_savings / 12 - monthly_cost)    print("=== GraphRAG ROI 분석 ===")    print(f"  연간 절감액: ${annual_savings:,.0f}")    print(f"  초기 비용:   ${initial_cost:,.0f}")    print(f"  월 운영비:   ${monthly_cost:,.0f}")    print(f"  연간 총비용: ${annual_cost:,.0f}")    print(f"  ROI: {roi:.1f}%")    print(f"  손익분기: {payback_months:.1f}개월")    return roi# 예시: 고객 응답 자동화calculate_roi(    annual_savings=120000,    initial_cost=30000,    monthly_cost=2500)

---## 3. PoC → 프로덕션 전환### 5단계 전환 프로세스| 단계 | 기간 | 목표 | 성공 기준 ||------|------|------|----------|| 1. PoC | 2주 | 기술 검증 | 3-hop 쿼리 작동 || 2. 파일럿 | 1개월 | 실사용자 테스트 | RAGAS > 0.7 || 3. 베타 | 2개월 | 안정성 검증 | P99 < 3초 || 4. GA | 1개월 | 전사 배포 | SLA 99.5% || 5. 운영 | 지속 | 모니터링 + 개선 | 월간 리뷰 |> **실패하는 PoC의 90%**: 데이터 품질 문제 (온톨로지 미정의, ER 미수행)

---## 4. 보안 & 컴플라이언스### PII 처리```pythondef mask_pii(text: str) -> str:    # 이름 → Person_A, Person_B    # 전화번호 → XXX-XXXX-XXXX    # 주민번호 → 완전 삭제    pass```### Neo4j RBAC (Enterprise)```cypherCREATE ROLE consultant;GRANT MATCH {*} ON GRAPH * NODES Product TO consultant;DENY MATCH {*} ON GRAPH * NODES Person TO consultant;```### 감사 로그| 로그 항목 | 용도 ||----------|------|| 쿼리 로그 | 누가 어떤 쿼리를 실행했는지 || 변경 로그 | 노드/관계 생성/수정/삭제 추적 || API 호출 로그 | LLM API 사용량 추적 |

In [ ]:
# PII 마스킹 함수import redef mask_pii(text: str) -> str:    # 전화번호 마스킹    text = re.sub(r'\d{3}-\d{4}-\d{4}', 'XXX-XXXX-XXXX', text)    # 이메일 마스킹    text = re.sub(r'[\w.-]+@[\w.-]+', '***@***.***', text)    return text# 테스트sample = "김철수 (010-1234-5678, kim@example.com)에게 연락하세요."print(f"원본: {sample}")print(f"마스킹: {mask_pii(sample)}")

---## 5. CI/CD + 운영 자동화### GitHub Actions 파이프라인```yamlname: GraphRAG CIon: [push, pull_request]jobs:  test:    runs-on: ubuntu-latest    services:      neo4j:        image: neo4j:5-community        env:          NEO4J_AUTH: neo4j/testpassword          NEO4J_PLUGINS: '["apoc"]'        ports: ["7687:7687"]    steps:      - uses: actions/checkout@v4      - run: pip install -r requirements.txt      - run: python -m pytest tests/ -v      - run: python eval/run_ragas.py --threshold 0.7```### Quality Gate| 체크 | 임계값 | 실패 시 ||------|--------|--------|| RAGAS 평균 | > 0.7 | 배포 차단 || P99 응답시간 | < 3초 | 경고 || 에러율 | < 1% | 배포 차단 |

In [ ]:
# 프로덕션 체크리스트 자가진단def production_checklist():    checks = []    # 1. Neo4j 연결    try:        driver.verify_connectivity()        checks.append(("Neo4j 연결", "PASS"))    except:        checks.append(("Neo4j 연결", "FAIL"))    # 2. 인덱스 확인    try:        result = run_query("SHOW INDEXES YIELD name RETURN count(*) AS cnt", print_result=False)        cnt = result[0]["cnt"]        checks.append(("인덱스", "PASS" if cnt >= 3 else "WARN"))    except:        checks.append(("인덱스", "FAIL"))    # 3. 데이터 존재    result = run_query("MATCH (n) RETURN count(n) AS cnt", print_result=False)    cnt = result[0]["cnt"] if result else 0    checks.append(("그래프 데이터", "PASS" if cnt > 0 else "FAIL"))    # 4. API 키    api_key = os.getenv("OPENAI_API_KEY", "")    checks.append(("OpenAI API 키", "PASS" if len(api_key) > 10 else "FAIL"))    print("=" * 50)    print("  프로덕션 체크리스트")    print("=" * 50)    for name, status in checks:        icon = {"PASS": "[OK]", "WARN": "[!!]", "FAIL": "[XX]"}[status]        print(f"  {icon} {name}")    print(f"\n  결과: {sum(1 for _,s in checks if s=='PASS')}/{len(checks)} 통과")production_checklist()

---## 6. 연습 문제### 문제 1: ROI 계산본인 도메인에서 GraphRAG 도입 시 ROI를 계산하세요.### 문제 2: CI/CD 파이프라인GitHub Actions 워크플로우 YAML을 작성하세요.### 문제 3: 보안 정책PII 마스킹 규칙을 3개 이상 추가하세요.

In [ ]:
# [연습] 여기에 코드를 작성하세요print("엔터프라이즈 GraphRAG 연습 문제를 풀어보세요!")

---## 7. 정리| 영역 | 핵심 ||------|------|| ROI | 절감액 vs 비용, 손익분기점 || PoC 전환 | 5단계 (PoC→파일럿→베타→GA→운영) || 보안 | PII 마스킹, RBAC, 감사 로그 || CI/CD | Quality Gate, 자동 테스트 |> **"GraphRAG 도입의 80%는 기술이 아니라 조직과 프로세스 문제다."**### 다음 Part 13: 캡스톤 프로젝트

In [ ]:
# 세션 정리# driver.close()# print("[OK] Neo4j 드라이버 종료 완료")print("실습을 마칩니다. 수고하셨습니다!")